In [ ]:
import pandas as pd
import plip_analysis as pa
from pathlib import Path

# Load the data

In [ ]:
crystal_interactions = {csv_path.stem: pa.PLIntReport.from_csv(csv_path) for csv_path in Path("mers_crystal_interactions").glob("*.csv")}
docked_interactions = {csv_path.stem: pa.PLIntReport.from_csv(csv_path) for csv_path in Path("mers_docked_interactions").glob("*.csv")}

In [ ]:
new_crystal_interactions = {}
for key, value in crystal_interactions.items():
    crystal_name = "-".join(key.split("_")[2].split("-")[:2])
    new_crystal_interactions[crystal_name] = value

In [ ]:
score_list = []
missing = []
for level in pa.FingerprintLevel:
    for name, docked_plint_report in docked_interactions.items():
        common_key = name.split("_")[0]
        crystal_plint_report = new_crystal_interactions.get(common_key)
        if crystal_plint_report is None:
            missing.append(name)
            continue
        score_list.append({'ASAP_Ligand_ID': common_key, 'Variant': 'MERS', **pa.InteractionScore.from_fingerprints(crystal_plint_report, docked_plint_report, level).dict()})

In [ ]:
df = pd.DataFrame.from_records(score_list)

In [ ]:
import plotly.express as px

In [ ]:
category_orders = {"provenance": [lvl.value for lvl in pa.FingerprintLevel]}
fig = px.bar(df.sort_values('ASAP_Ligand_ID'), 
             x='ASAP_Ligand_ID', 
             y='tversky_index',
             color='provenance', 
             barmode='group', 
             category_orders=category_orders, 
             title='MERS Interaction Scores',
             template='simple_white',
             height=600, width=1600)

In [ ]:
fig.show()

In [ ]:
fig.write_image("mers_interaction_scores.png")